# Nano World Model — Colab Quickstart

Run a pretrained NanoWM checkpoint on Google Colab's free T4 GPU and watch the resulting rollout video. No local GPU required.

Uses the smallest available demo (DINO-WM **Point Maze**, NanoWM-B/2, 30k-step checkpoint). Other DINO-WM envs (PushT, Wall, Rope, Granular) follow the same pattern — swap the HuggingFace repo id in step 6 and grab the matching folder from OSF in step 8.

## Before you start

1. `Runtime → Change runtime type → T4 GPU`
2. Manually grab the Point Maze dataset (OSF has no scriptable download):
   - Open https://osf.io/bmw48/?view_only=a56a296ce3b24cceaf408383a175ce28
   - Download the `point_maze/` folder as a zip
   - Upload it to Colab at `/content/point_maze.zip` (use the folder icon in the left sidebar → upload)
3. Run cells top to bottom. **One manual restart** is required after step 3 — instructions in that cell.

Expected total runtime: ~15 minutes (5 min setup + 5 min unzip + 3-7 min rollout).

## Step 1 — GPU check

Should show a Tesla T4. If not, change runtime type.

In [ ]:
!nvidia-smi | head -20

## Step 2 — Clone the repo

Idempotent — safe to re-run.

In [ ]:
import os
if not os.path.isdir("/content/nano-world-model"):
    !git clone -q https://github.com/simchowitzlabpublic/nano-world-model.git /content/nano-world-model
%cd /content/nano-world-model
!pwd

## Step 3 — Install pinned dependencies

The repo targets `diffusers==0.24.0` (Nov 2023). Modern Colab ships much newer `huggingface_hub` / `transformers` / `accelerate` that break the old diffusers imports. The pins below restore a compatible set from that era.

Takes ~3-5 minutes. Ignore pip dependency-resolver warnings — they're about unrelated Colab packages (gradio, peft, datasets, sentence-transformers) which we don't use.

In [ ]:
!pip install -q \
  "huggingface_hub==0.20.3" "transformers==4.36.0" "diffusers[torch]==0.24.0" \
  "accelerate==0.25.0" \
  timm einops \
  pytorch-lightning==1.9.5 torchmetrics \
  omegaconf hydra-core \
  decord h5py \
  opencv-python imageio imageio-ffmpeg moviepy==1.0.3 scikit-image av \
  sentencepiece safetensors

## ⚠️ Step 3.5 — Restart the runtime

**Required.** Colab pre-imports `huggingface_hub` on session start, so the old version stays cached in Python memory until you restart.

1. Click `Runtime → Restart session` (NOT "Restart and run all")
2. After restart, **start from Step 4 below** — do NOT re-run steps 1–3. Cloned repo + installed packages survive the restart.

If you skip this, you'll get `ImportError: cannot import name 'hf_cache_home'` or similar later.

## Step 4 — Restore working directory

Runtime restart resets `cwd` to `/content/`. Put it back.

In [ ]:
%cd /content/nano-world-model
!pwd && ls src/sample/rollout.py

## Step 5 — Patch for PyTorch 2.6+ compatibility

PyTorch 2.6 (Jan 2025) changed `torch.load`'s default for `weights_only` from `False` to `True`, which rejects the official checkpoints. Patch the one line in `find_model` to pass `weights_only=False` explicitly. Safe — the checkpoints come from the project's HuggingFace org.

Idempotent: if a future repo update has already merged this fix, the cell prints `already patched` and continues.

In [ ]:
path = "/content/nano-world-model/src/utils/nanowm_utils.py"
old = "torch.load(model_name, map_location=lambda storage, loc: storage)"
new = "torch.load(model_name, map_location=lambda storage, loc: storage, weights_only=False)"

with open(path) as f:
    code = f.read()

if new in code:
    print("Already patched.")
elif old in code:
    with open(path, "w") as f:
        f.write(code.replace(old, new))
    print("Patched.")
else:
    raise RuntimeError("Target line not found in nanowm_utils.py — check manually.")

## Step 6 — Download the pretrained checkpoint

Pulls `model.safetensors` + `config.yaml` from HuggingFace. ~600 MB.

In [ ]:
from huggingface_hub import snapshot_download

ckpt_dir = snapshot_download(
    repo_id="knightnemo/nanowm-b2-dino-wm-point-maze-30k",
    local_dir="/content/checkpoints/point_maze",
)
print("Downloaded to:", ckpt_dir)
!ls -lah /content/checkpoints/point_maze

## Step 7 — Convert safetensors → .pt and locate config

`find_model` uses `torch.load`, which can't read `.safetensors` directly. Convert once and cache `model.pt` next to the original.

Sets `CKPT_PATH` and `CONFIG_PATH`. Re-run this cell if you ever restart the runtime — Python variables don't survive restart.

In [ ]:
import os, glob, torch
from safetensors.torch import load_file

ckpt_dir = "/content/checkpoints/point_maze"
safetensor_files = glob.glob(f"{ckpt_dir}/**/*.safetensors", recursive=True)
config_files     = glob.glob(f"{ckpt_dir}/**/config.yaml", recursive=True)

assert safetensor_files, f"No .safetensors in {ckpt_dir} — re-run step 6."
assert config_files, f"No config.yaml in {ckpt_dir} — re-run step 6."

src = safetensor_files[0]
dst = src.replace(".safetensors", ".pt")

# Always re-derive to avoid any chance of stale/corrupt .pt files between sessions
if os.path.exists(dst):
    os.remove(dst)
print(f"Converting {src} -> {dst}")
torch.save(load_file(src), dst)
_ = torch.load(dst, weights_only=False)  # sanity-check round-trip

CKPT_PATH = dst
CONFIG_PATH = config_files[0]
print("\nCKPT_PATH   =", CKPT_PATH)
print("CONFIG_PATH =", CONFIG_PATH)
print("✓ Checkpoint converted and verified loadable.")

## Step 8 — Unzip the Point Maze dataset

Assumes you've uploaded `point_maze.zip` to `/content/point_maze.zip` via Colab's left-sidebar file upload icon (folder → upload).

Unzip can take 2-5 minutes — many small frame files.

In [ ]:
ZIP_PATH = "/content/point_maze.zip"
assert os.path.isfile(ZIP_PATH), (
    f"Couldn't find {ZIP_PATH}. Upload point_maze.zip via Colab's left-sidebar "
    f"file upload icon (folder → upload) into /content/."
)
!ls -lh "$ZIP_PATH"

os.makedirs("/content/dino_wm_data", exist_ok=True)

# Skip the unzip if it's already done (safe to re-run this cell)
if os.path.isdir("/content/dino_wm_data/point_maze") and os.listdir("/content/dino_wm_data/point_maze"):
    print("Already unzipped — skipping.")
else:
    !unzip -q -o "$ZIP_PATH" -d /content/dino_wm_data/

# Some zips contain an extra nested point_maze/ — flatten if so
if os.path.isdir("/content/dino_wm_data/point_maze/point_maze"):
    print("Flattening nested point_maze/ directory.")
    !mv /content/dino_wm_data/point_maze/point_maze/* /content/dino_wm_data/point_maze/ 2>/dev/null
    !rmdir /content/dino_wm_data/point_maze/point_maze 2>/dev/null

print("\nContents of /content/dino_wm_data/point_maze/ (first 10):")
!ls /content/dino_wm_data/point_maze | head -10

## Step 9 — Run the rollout

Known-good Point Maze settings:
- `--history_length 3` (must be < `num_frames=4` from the model config)
- `--rollout_length 20` (Point Maze trajectories are ~100-200 frames, `frame_interval=5` → needs total ≤ ~30)
- `--num_sampling_steps 15` (fast preview; bump to 50 for better quality, 250 for paper-grade)

Expected runtime: **3-7 minutes** on T4. Progress bar will tick `0/17 → 17/17` as new frames are generated. The first step is the slowest due to CUDA kernel warmup.

If it looks stuck, open a new cell and run `!nvidia-smi` — high GPU utilization means it's grinding away normally.

In [ ]:
os.environ["DATASET_DIR"] = "/content/dino_wm_data"
os.environ["RESULTS_DIR"] = "/content/results"
os.environ["CKPT_PATH"]   = CKPT_PATH
os.environ["CONFIG_PATH"] = CONFIG_PATH

%cd /content/nano-world-model

!python src/sample/rollout.py \
    --config "$CONFIG_PATH" \
    --ckpt   "$CKPT_PATH" \
    --save_path /content/results/point_maze_demo \
    --num_samples 2 --batch_size 2 \
    --rollout_length 20 --history_length 3 \
    --num_sampling_steps 15 --scheduling_mode sequential \
    --history_stabilization_level 0.02 --fps 8

## Step 10 — Watch the result

Three videos per sample:
- `sample_XXXX_gen.mp4` — model's prediction
- `sample_XXXX_gt.mp4` — ground truth (same actions)
- `sample_XXXX_compare.mp4` — side-by-side

In [ ]:
out_dir = "/content/results/point_maze_demo"
files = sorted(glob.glob(f"{out_dir}/*.mp4"))
assert files, f"No videos in {out_dir}. Step 9 must have errored — scroll up to find the trace."
print(f"Found {len(files)} mp4 files:")
for f in files:
    print(" -", os.path.basename(f), f"({os.path.getsize(f)/1024:.1f} KB)")

In [ ]:
from IPython.display import HTML
from base64 import b64encode

def show_mp4(path, width=480):
    data = b64encode(open(path, "rb").read()).decode()
    return HTML(f'<video width={width} controls loop autoplay muted><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

compare = f"{out_dir}/sample_0000_compare.mp4"
show_mp4(compare) if os.path.isfile(compare) else show_mp4(files[0])

## Next steps

- **Better quality**: `--num_sampling_steps 50` (or 250 for paper-grade). Slower.
- **Longer rollouts**: bump `--rollout_length`. Max useful ~30 for Point Maze (`frame_interval=5`, trajectories ~150 frames).
- **Other DINO-WM envs**: change the HuggingFace repo id in step 6 (`nanowm-b2-dino-wm-wall-15k`, `nanowm-b2-dino-wm-pusht-100k`, etc.) and the OSF dataset folder in step 8.
- **CSGO demo** (the impressive one): can't run on free Colab — dataset is ~675 GB.

### Common failure modes

| Symptom | Fix |
|---|---|
| `No such file: src/sample/rollout.py` | cwd reset. Re-run step 4. |
| `NameError: CKPT_PATH` / `CONFIG_PATH` | Restart wiped them. Re-run step 7. |
| `ImportError: hf_cache_home` / `is_offline_mode` / `split_torch_state_dict_into_shards` | Didn't restart after step 3. Restart now (cells survive). |
| `UnpicklingError: Weights only load failed` | Step 5 didn't apply. Re-run it. |
| `num_frames (4) must be > n_context_frames (4)` | Don't change `--history_length 3` (Point Maze's model has `num_frames=4`). |
| `Only found 0 valid slices` | `--rollout_length` is too large for Point Maze. Keep at 20-30. |